In [3]:
# Célula 1: Setup e Conexão com PostgreSQL
import pandas as pd
import numpy as np
from pathlib import Path
import psycopg2
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Caminhos
PROC = Path("../data/processed")

# ── Conexão com PostgreSQL ────────────────────────────────────────────────
DB_USER = "postgres"
DB_PASS = quote_plus("An@Clara10")
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "pecuaria"

# Engine SQLAlchemy (usamos pra carregar os DataFrames)
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Testa conexão
with engine.connect() as conn:
    resultado = conn.execute(text("SELECT current_database()"))
    banco = resultado.fetchone()[0]
    print(f"Conectado ao banco: {banco}")

Conectado ao banco: pecuaria


In [5]:
# Célula 2: Carga de Dados
tabelas = ["animais", "estoque", "performance", "pesagens", "sanitario"]

print("--- INICIANDO CARGA DE DADOS ---")

for tabela in tabelas:
    try:
        caminho_csv = PROC / f"{tabela}.csv"
        df = pd.read_csv(caminho_csv, dtype={'brinco': str})

        for col in df.columns:
            if 'data' in col.lower():
                df[col] = pd.to_datetime(df[col])

        df.to_sql(tabela, con=engine, if_exists='replace', index=False)

        print(f"Tabela '{tabela}' carregada ({len(df)} linhas).")

    except Exception as e:
        print(f"Erro ao carregar {tabela}: {e}")

print("\n--- CARGA FINALIZADA ---")

--- INICIANDO CARGA DE DADOS ---
Tabela 'animais' carregada (500 linhas).
Tabela 'estoque' carregada (540 linhas).
Tabela 'performance' carregada (500 linhas).
Tabela 'pesagens' carregada (4148 linhas).
Tabela 'sanitario' carregada (6181 linhas).

--- CARGA FINALIZADA ---


In [10]:
# Célula 3: Salvando estrutura de chaves
# Definido o caminho da pasta SQL
SQL_FOLDER = Path("../sql")
SQL_FOLDER.mkdir(parents=True, exist_ok=True)

# Definindo o conteúdo do script de integridade
# Criação de chaves primárias e estrangeiras
script_integridade = """
-- Script de Relacionamentos e Chaves Estrangeiras
-- Garante que o banco esteja íntegro

-- Relacionando Pesagens com Animais
ALTER TABLE pesagens
ADD CONSTRAINT fk_animais_pesagens
FOREIGN KEY (id_animal) REFERENCES animais (id_animal) ON DELETE CASCADE;

-- Relacionando Sanitário com Animais
ALTER TABLE sanitario
ADD CONSTRAINT fk_animais_sanitario
FOREIGN KEY (id_animal) REFERENCES animais (id_animal) ON DELETE CASCADE;

-- Relacionando Performance com Animais
ALTER TABLE performance
ADD CONSTRAINT fk_animais_performance
FOREIGN KEY (id_animal) REFERENCES animais (id_animal) ON DELETE CASCADE;
"""

# Salvando o arquivo na pasta
file_path = SQL_FOLDER / "01_setup_integridade.sql"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(script_integridade)

print(f"Arquivo SQL criado em: {file_path}")

Arquivo SQL criado em: ..\sql\01_setup_integridade.sql


In [11]:
# Célula 4: Salvando estruturas de tabelas
# Definindo o script de criação das tabelas
script_criacao = """
-- Script de Criação das Tabelas (DDL)
-- Gerado com base nos dados reais de simulação

CREATE TABLE IF NOT EXISTS animais (
    id_animal INT PRIMARY KEY,
    brinco VARCHAR(20) UNIQUE NOT NULL,
    raca VARCHAR(50),
    lote INT,
    peso_entrada FLOAT,
    gmd_base FLOAT,
    data_entrada DATE,
    status VARCHAR(50)
);

CREATE TABLE IF NOT EXISTS pesagens (
    id_pesagem INT PRIMARY KEY,
    id_animal INT,
    brinco VARCHAR(20),
    raca VARCHAR(50),
    lote INT,
    data_pesagem DATE,
    dias_conf INT,
    peso_kg FLOAT,
    gmd_periodo FLOAT,
    arroba_atual FLOAT,
    preco_arroba_rs FLOAT,
    valor_mercado_rs FLOAT
);

CREATE TABLE IF NOT EXISTS sanitario (
    id_evento INT PRIMARY KEY,
    id_animal INT,
    brinco VARCHAR(20),
    data_evento DATE,
    tipo VARCHAR(50),
    produto VARCHAR(100),
    dose_ml FLOAT,
    custo_unit FLOAT,
    custo_total FLOAT
);

CREATE TABLE IF NOT EXISTS estoque (
    id_estoque INT PRIMARY KEY,
    lote INT,
    n_animais INT,
    data_ref DATE,
    insumo VARCHAR(50),
    consumo_kg FLOAT,
    preco_rs_kg FLOAT,
    custo_total_rs FLOAT,
    estoque_kg FLOAT
);

CREATE TABLE IF NOT EXISTS performance (
    id_animal INT PRIMARY KEY,
    brinco VARCHAR(20),
    raca VARCHAR(50),
    lote INT,
    peso_entrada FLOAT,
    gmd_base FLOAT,
    data_entrada DATE,
    status VARCHAR(50),
    data_pesagem DATE,
    peso_kg FLOAT,
    gmd_periodo FLOAT,
    arroba_atual FLOAT,
    preco_arroba_rs FLOAT,
    valor_mercado_rs FLOAT,
    dias_conf INT,
    custo_sanitario_rs FLOAT,
    custo_racao_lote_rs FLOAT,
    n_animais_lote INT,
    custo_racao_animal_rs FLOAT,
    gmd_total FLOAT,
    ganho_peso_kg FLOAT,
    custo_total_animal_rs FLOAT,
    custo_por_kg_ganho FLOAT,
    margem_bruta_rs FLOAT,
    conversao_alimentar FLOAT,
    classe_performance VARCHAR(50)
);
"""

# Salvando na pasta SQL
file_path_create = SQL_FOLDER / "00_create_tables.sql"
with open(file_path_create, "w", encoding="utf-8") as f:
    f.write(script_criacao)

print(f"Arquivo de criação das tabelas gerado em: {file_path_create}")

Arquivo de criação das tabelas gerado em: ..\sql\00_create_tables.sql
